In [0]:
# ============================================================
# DELETE OLD SPLIT FILES
# ============================================================

volume_path = "/Volumes/adtech_catalog/bronze/landing_zone/"

try:
    dbutils.fs.rm(volume_path + "train_split.csv")
    print("Deleted: train_split.csv")
except Exception as e:
    print(f"train_split.csv not found or already deleted: {e}")

try:
    dbutils.fs.rm(volume_path + "test_split.csv")
    print("Deleted: test_split.csv")
except Exception as e:
    print(f"test_split.csv not found or already deleted: {e}")

try:
    dbutils.fs.rm(volume_path + "split_metadata.csv")
    print("Deleted: split_metadata.csv")
except Exception as e:
    print(f"split_metadata.csv not found or already deleted: {e}")

print("\nOld split files deleted. Now re-run 01_EDA_and_Data_Preparation.py")

Deleted: train_split.csv
Deleted: test_split.csv
Deleted: split_metadata.csv

Old split files deleted. Now re-run 01_EDA_and_Data_Preparation.py


In [0]:
# Databricks notebook source
# ============================================================
# 01_EDA_and_Data_Preparation
# ============================================================
# Purpose: Exploratory data analysis and train/test split
# Author: Sanju
# Team: 5 Members
# Date: 2026-07-31
#
# This notebook performs EDA and creates a fixed train/test split
# that will be reused across all ML models.
#


import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import yaml
import os
import warnings
warnings.filterwarnings("ignore")

spark = SparkSession.builder.getOrCreate()

print("="*70)
print("EDA AND DATA PREPARATION")
print("="*70)

# ============================================================
# 1. LOAD CONFIGURATION
# ============================================================

def load_yaml_config():
    """Load pipeline configuration from YAML file"""
    try:
        try:
            with open("pipeline_manifest.yaml", "r") as f:
                config = yaml.safe_load(f)
                print("Loaded config from local path")
                return config
        except:
            pass

        try:
            config_path = "/Volumes/adtech_catalog/bronze/landing_zone/pipeline_manifest.yaml"
            try:
                dbutils.fs.ls(config_path)
                config_content = dbutils.fs.head(config_path)
                config = yaml.safe_load(config_content)
                print(f"Loaded config from: {config_path}")
                return config
            except:
                print("Config file not found in DBFS")
                return None
        except:
            return None

    except Exception as e:
        print(f"Could not load config: {e}")
        return None

config = load_yaml_config()

ENVIRONMENT = config.get('environment', 'development') if config else 'development'
VERSION = datetime.now().strftime("%Y%m%d_%H%M%S")
GIT_COMMIT = os.environ.get('GIT_COMMIT', 'local')

# Split ratio from config or default
SPLIT_RATIO = config.get('ml_pipeline', {}).get('test_split_ratio', 0.2) if config else 0.2
RANDOM_SEED = config.get('ml_pipeline', {}).get('random_seed', 42) if config else 42

print("="*70)
print("CONFIGURATION SUMMARY")
print("="*70)
print(f"Environment: {ENVIRONMENT}")
print(f"Version: {VERSION}")
print(f"Git Commit: {GIT_COMMIT}")
print(f"Split Ratio: {SPLIT_RATIO}")
print(f"Random Seed: {RANDOM_SEED}")
print("="*70)

# ============================================================
# 2. LOAD GOLD DATA WITH IDEMPOTENCY CHECK
# ============================================================

print("\nLOADING GOLD DATA...")

def check_table_exists(table_name):
    try:
        spark.sql(f"DESCRIBE {table_name}")
        return True
    except:
        return False

def check_s3_path(s3_path):
    try:
        dbutils.fs.ls(s3_path)
        return True
    except:
        return False

GOLD_TABLE = "adtech_catalog.gold.fact_ad_performance"
S3_PATH = "s3://adtech-optimizer-data/gold/fact_ad_performance/"

df_gold = None

# Try S3 first (if available)
if check_s3_path(S3_PATH):
    print(f"Loading from S3: {S3_PATH}")
    df_spark = spark.read.parquet(S3_PATH)
    df_gold = df_spark.toPandas()
    print(f"Loaded {len(df_gold):,} rows from S3")
else:
    print(f"S3 path not found, trying table: {GOLD_TABLE}")
    if check_table_exists(GOLD_TABLE):
        df_spark = spark.table(GOLD_TABLE)
        df_gold = df_spark.toPandas()
        print(f"Loaded {len(df_gold):,} rows from table")
    else:
        print(f"ERROR: Table {GOLD_TABLE} does not exist.")
        print("Please run 04_FEATURE_ENGINEERING.py first.")
        dbutils.notebook.exit("Gold table not found")

if df_gold is None or len(df_gold) == 0:
    print("ERROR: No data loaded.")
    dbutils.notebook.exit("No data available")

# Convert timestamp columns to string to avoid Arrow conversion issues
timestamp_cols = ['ingestion_timestamp', 'processing_date']
for col in timestamp_cols:
    if col in df_gold.columns:
        df_gold[col] = df_gold[col].astype(str) if col == 'ingestion_timestamp' else pd.to_datetime(df_gold[col])

# Fill nulls
df_gold = df_gold.fillna(0)

print(f"Data shape: {df_gold.shape}")
print(f"Columns: {len(df_gold.columns)}")

# ============================================================
# 3. DATA OVERVIEW
# ============================================================

print("\n" + "="*70)
print("DATA OVERVIEW")
print("="*70)

print("\nFirst 5 rows:")
display(df_gold.head(5))

print("\nData types:")
print(df_gold.dtypes.value_counts())

print("\nBasic statistics:")
numeric_cols_for_stats = df_gold.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols_for_stats:
    display(df_gold[numeric_cols_for_stats].describe())

# ============================================================
# 4. TARGET DISTRIBUTIONS
# ============================================================

print("\n" + "="*70)
print("TARGET DISTRIBUTIONS")
print("="*70)

targets = ['ctr', 'roas', 'conversion_rate', 'high_performance']

for target in targets:
    if target in df_gold.columns:
        print(f"\n{target.upper()} Distribution:")
        print(f"  Count: {df_gold[target].count():,}")
        print(f"  Mean: {df_gold[target].mean():.4f}")
        print(f"  Min: {df_gold[target].min():.4f}")
        print(f"  Max: {df_gold[target].max():.4f}")
        print(f"  Std: {df_gold[target].std():.4f}")

# Check class balance for high_performance
if 'high_performance' in df_gold.columns:
    hp_counts = df_gold['high_performance'].value_counts()
    total = len(df_gold)
    print(f"\nHigh Performance Class Balance:")
    print(f"  0 (Not Profitable): {hp_counts.get(0, 0):,} ({hp_counts.get(0, 0)/total*100:.1f}%)")
    print(f"  1 (Profitable):     {hp_counts.get(1, 0):,} ({hp_counts.get(1, 0)/total*100:.1f}%)")
    
    # Class Imbalance Recommendations
    print("\n" + "-"*50)
    print("CLASS IMBALANCE RECOMMENDATIONS")
    print("-"*50)
    imbalance_ratio = hp_counts.get(0, 0) / hp_counts.get(1, 1) if hp_counts.get(1, 0) > 0 else 0
    print(f"  Imbalance Ratio: {imbalance_ratio:.1f}:1")
    print("  Recommended approaches for classification models:")
    print("    1. Use class_weight='balanced' in RandomForest/GradientBoosting")
    print("    2. Use scale_pos_weight in XGBoost")
    print("    3. Use SMOTE for oversampling (with caution on 1,000 rows)")
    print("    4. Use StratifiedKFold for cross-validation")

# ============================================================
# 5. TRAIN/TEST SPLIT (STRATIFIED WITH GUARANTEE)
# ============================================================

print("\n" + "="*70)
print("TRAIN/TEST SPLIT")
print("="*70)

# Check class distribution before split
hp_counts = df_gold['high_performance'].value_counts()
print(f"Class distribution in full dataset:")
print(f"  0 (Not Profitable): {hp_counts.get(0, 0):,}")
print(f"  1 (Profitable):     {hp_counts.get(1, 0):,}")

def ensure_both_classes_in_train(df, test_size=0.2, max_attempts=20):
    """
    Find a random split that ensures both classes appear in training.
    If not found after max_attempts, manually move one sample from test to train.
    """
    stratify_col = df['high_performance']
    
    for attempt in range(max_attempts):
        seed = RANDOM_SEED + attempt
        train_df, test_df = train_test_split(
            df,
            test_size=test_size,
            random_state=seed,
            stratify=stratify_col
        )
        
        # Check if both classes exist in training
        if len(np.unique(train_df['high_performance'])) == 2:
            print(f"  Valid split found with random_seed={seed}")
            return train_df, test_df, seed
    
    # If we reach here, no valid split was found
    print("  WARNING: Could not find a valid split with both classes.")
    print("  Manually ensuring both classes in training...")
    
    # Try one more split without stratify
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=RANDOM_SEED
    )
    
    # Check if both classes exist in training
    if len(np.unique(train_df['high_performance'])) == 2:
        print(f"  Valid split found without stratify")
        return train_df, test_df, RANDOM_SEED
    
    # Manual fix: Move one profitable sample from test to train
    profitable_indices = df[df['high_performance'] == 1].index.tolist()
    if len(profitable_indices) > 0:
        # Take one profitable ad
        move_idx = profitable_indices[0]
        train_df = pd.concat([train_df, df.loc[[move_idx]]])
        test_df = test_df[test_df.index != move_idx]
        print(f"  Manually moved 1 profitable ad from test to training.")
        print(f"  New training size: {len(train_df):,}")
        print(f"  New test size: {len(test_df):,}")
    
    return train_df, test_df, RANDOM_SEED

train_df, test_df, used_seed = ensure_both_classes_in_train(df_gold, SPLIT_RATIO)

print(f"\nSplit Summary:")
print(f"  Used Random Seed: {used_seed}")
print(f"  Training rows: {len(train_df):,} ({len(train_df)/len(df_gold)*100:.1f}%)")
print(f"  Test rows: {len(test_df):,} ({len(test_df)/len(df_gold)*100:.1f}%)")

# Verify class balance in split
if 'high_performance' in df_gold.columns:
    train_hp = train_df['high_performance'].value_counts()
    test_hp = test_df['high_performance'].value_counts()
    
    print(f"\nClass balance in Training:")
    print(f"  0 (Not Profitable): {train_hp.get(0, 0):,} ({train_hp.get(0, 0)/len(train_df)*100:.1f}%)")
    print(f"  1 (Profitable):     {train_hp.get(1, 0):,} ({train_hp.get(1, 0)/len(train_df)*100:.1f}%)")
    
    print(f"\nClass balance in Test:")
    print(f"  0 (Not Profitable): {test_hp.get(0, 0):,} ({test_hp.get(0, 0)/len(test_df)*100:.1f}%)")
    print(f"  1 (Profitable):     {test_hp.get(1, 0):,} ({test_hp.get(1, 0)/len(test_df)*100:.1f}%)")
    
    # Verify both classes exist in training
    if len(np.unique(train_df['high_performance'])) < 2:
        print("\n  ERROR: Training set still has only one class!")
        print("  Manual intervention required.")
    else:
        print("\n  BOTH CLASSES PRESENT IN TRAINING SET")

# ============================================================
# 6. SAVE SPLIT INDICES
# ============================================================

print("\nSAVING SPLIT INDICES...")

volume_path = "/Volumes/adtech_catalog/bronze/landing_zone/"

# DELETE OLD FILES FIRST
try:
    dbutils.fs.rm(volume_path + "train_split.csv")
    print("Removed old train_split.csv")
except:
    pass

try:
    dbutils.fs.rm(volume_path + "test_split.csv")
    print("Removed old test_split.csv")
except:
    pass

# Create a copy without timestamp columns for saving
train_save = train_df.copy()
test_save = test_df.copy()

# Drop problematic timestamp columns before saving
for col in ['ingestion_timestamp', 'processing_date']:
    if col in train_save.columns:
        train_save = train_save.drop(columns=[col])
    if col in test_save.columns:
        test_save = test_save.drop(columns=[col])

# Add split indicator column
train_save['split'] = 'train'
test_save['split'] = 'test'

try:
    train_save.to_csv(volume_path + "train_split.csv", index=False)
    test_save.to_csv(volume_path + "test_split.csv", index=False)
    print(f"Training split saved to: {volume_path}train_split.csv")
    print(f"Test split saved to: {volume_path}test_split.csv")
except Exception as e:
    print(f"Error saving to Volume: {e}")

# ============================================================
# 7. CORRELATION ANALYSIS
# ============================================================

print("\n" + "="*70)
print("CORRELATION ANALYSIS")
print("="*70)

# Select numeric columns for correlation
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

# Limit to key columns for readability
key_features = ['ctr', 'roas', 'conversion_rate', 'high_performance',
                'cost_per_click', 'total_impressions', 'total_clicks',
                'avg_watch_ratio', 'avg_ded_score']

available_features = [col for col in key_features if col in numeric_cols]

if len(available_features) > 1:
    corr_matrix = train_df[available_features].corr()
    print("\nCorrelation Matrix:")
    display(corr_matrix)

    # Print top correlations with target
    for target in ['ctr', 'roas', 'conversion_rate']:
        if target in corr_matrix.columns:
            print(f"\nTop correlations with {target}:")
            corr_target = corr_matrix[target].sort_values(ascending=False)
            for feature, corr in corr_target.items():
                if feature != target:
                    print(f"  {feature}: {corr:.3f}")

# ============================================================
# 8. CATEGORICAL FEATURE ANALYSIS
# ============================================================

print("\n" + "="*70)
print("CATEGORICAL FEATURE ANALYSIS")
print("="*70)

categorical_cols = ['ad_category', 'ad_device', 'ad_type', 'ad_location']

for col in categorical_cols:
    if col in train_df.columns:
        print(f"\n{col.upper()} Distribution:")
        value_counts = train_df[col].value_counts()
        print(value_counts.to_string())

# ============================================================
# 9. PRE-LAUNCH FEATURES (FOR ML)
# ============================================================

print("\n" + "="*70)
print("PRE-LAUNCH FEATURES")
print("="*70)

pre_launch_features = [
    "cost_per_click",
    "ad_video_length",
    "ad_category",
    "ad_device",
    "ad_type",
    "ad_location",
    "avg_ded_score",
    "category_age_affinity"
]

print("Features available before ad launch:")
for i, feat in enumerate(pre_launch_features, 1):
    print(f"  {i}. {feat}")

print(f"\nTotal pre-launch features: {len(pre_launch_features)}")

# ============================================================
# 10. SAVE VERSION HISTORY
# ============================================================

print("\nSAVING VERSION HISTORY...")

try:
    version_info = spark.createDataFrame([(
        VERSION,
        ENVIRONMENT,
        GIT_COMMIT,
        datetime.now().isoformat(),
        "EDA and Data Preparation",
        "SUCCESS"
    )], [
        "version_id",
        "environment",
        "git_commit",
        "deployed_at",
        "description",
        "status"
    ])

    version_info.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable("adtech_catalog.monitoring.version_history")

    print("Version history updated: adtech_catalog.monitoring.version_history")
    print(f"   Version: {VERSION}")

except Exception as e:
    print(f"Could not save version history: {e}")

# ============================================================
# 11. SUMMARY
# ============================================================

print("\n" + "="*70)
print("EDA AND DATA PREPARATION COMPLETE")
print("="*70)

print(f"""
SUMMARY
======================================================================
Version: {VERSION}
Environment: {ENVIRONMENT}

Data:
   - Total Rows: {len(df_gold):,}
   - Total Columns: {len(df_gold.columns)}
   - Pre-launch Features: {len(pre_launch_features)}

Split:
   - Method: Random Stratified (guaranteed both classes in training)
   - Used Random Seed: {used_seed}
   - Training: {len(train_df):,} rows ({len(train_df)/len(df_gold)*100:.1f}%)
   - Test: {len(test_df):,} rows ({len(test_df)/len(df_gold)*100:.1f}%)

Class Balance:
   - Profitable Ads (Train): {train_df['high_performance'].mean()*100:.1f}%
   - Profitable Ads (Test): {test_df['high_performance'].mean()*100:.1f}%

Saved Files:
   - {volume_path}train_split.csv
   - {volume_path}test_split.csv
   - {volume_path}split_metadata.csv

Next Steps:
   1. Run: 02_Baseline_Models.py
   2. Run: 09_Analytics_Only.py
======================================================================
""")

print("")

EDA AND DATA PREPARATION
Loaded config from: /Volumes/adtech_catalog/bronze/landing_zone/pipeline_manifest.yaml
CONFIGURATION SUMMARY
Environment: development
Version: 20260802_120127
Git Commit: local
Split Ratio: 0.2
Random Seed: 42

LOADING GOLD DATA...
Loading from S3: s3://adtech-optimizer-data/gold/fact_ad_performance/
Loaded 9,999 rows from S3
Data shape: (9999, 51)
Columns: 51

DATA OVERVIEW

First 5 rows:


Ad_Reference_ID,ad_category,ad_device,ad_location,ad_type,ad_type_catalog,cost_per_click,ad_video_length,total_clicks,total_impressions,ctr,avg_watch_duration,total_ad_spend,total_revenue,roas,total_conversions,conversion_rate,overall_conversion_rate,avg_user_age,unique_users,avg_watch_ratio,avg_ded_score,category_age_affinity,platforms_used,devices_used,platform_avg_roas,platform_total_spend,platform_total_revenue,active_time_slots,best_day,avg_hour,ingestion_date,ingestion_timestamp,engagement_efficiency,profit_margin,cost_per_conversion,high_performance,cost_efficiency_score,engagement_score,audience_alignment_score,ad_age_days,ad_lifecycle_stage,location_type,season,processing_date,export_timestamp,export_version,environment,year,month,day
AD_117417,Electronics,Tablet,Karnataka,Text,Image,1.88,0.0,2,215,0.009302325581395349,0.015813953488372095,3.76,0.0,0.0,0,0.0,0.0,42.07906976744186,215,0.0,0.09999999999999992,0.016279391278817973,"List(google, instagram, Unknown, facebook)","List(Tablet, Mobile, Desktop, Unknown)",0.004139926091193895,10036.52,2810.527602477398,"List(Afternoon, Evening_Prime, Late_Night, Morning)",5,11.962790697674418,2026-08-02,2026-08-02 10:11:18.245398,0.0,-1.0,0E-9,0,0.0,0.0037209302325581397,0.001627939127881796,0,New,Urban,Winter,2026-08-02T00:00:00.000Z,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_521967,Electronics,All-Devices,Delhi,Text,Video,3.16,30.0,4,196,0.02040816326530612,5.902551020408162,12.64,28.942578674033378,2.2897609710469444,1,0.25,0.00510204081632653,41.94387755102041,195,0.19675170068027212,0.09999999999999994,0.016370775937616752,"List(instagram, facebook, Unknown, google)","List(Unknown, Mobile, Tablet, Desktop)",0.004139926091193895,10036.52,2810.527602477398,"List(Late_Night, Morning, Afternoon, Evening_Prime)",3,10.392857142857142,2026-08-02,2026-08-02 10:11:18.245398,0.004015340830209635,1.2897609710469444,12.640000000,1,0.001614569878584345,0.1421887755102041,0.0016370775937616742,0,New,Urban,Spring,2026-08-02T00:00:00.000Z,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_578928,Electronics,Desktop,Karnataka,Video,Image,3.21,0.0,4,195,0.020512820512820513,0.03641025641025641,12.84,0.0,0.0,0,0.0,0.0,40.15384615384615,195,0.0,0.09999999999999995,0.01682097602891295,"List(facebook, google, instagram, Unknown)","List(Mobile, Tablet, Desktop, Unknown)",0.004139926091193895,10036.52,2810.527602477398,"List(Late_Night, Afternoon, Evening_Prime, Morning)",6,11.189743589743589,2026-08-02,2026-08-02 10:11:18.245398,0.0,-1.0,0E-9,0,0.0,0.008205128205128205,0.0016820976028912942,0,New,Urban,Winter,2026-08-02T00:00:00.000Z,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_512417,Electronics,Desktop,Delhi,Image,Video,3.44,30.0,2,187,0.0106951871657754,5.638502673796791,6.88,0.0,0.0,0,0.0,0.0,42.05882352941177,186,0.18795008912655975,0.09999999999999992,0.016483084258649586,"List(instagram, facebook, Unknown, google)","List(Mobile, Tablet, Unknown, Desktop)",0.004139926091193895,10036.52,2810.527602477398,"List(Late_Night, Evening_Prime, Afternoon, Morning)",7,11.101604278074866,2026-08-02,2026-08-02 10:11:18.245398,0.0020101613810327244,-1.0,0E-9,0,0.0,0.06066310160427808,0.0016483084258649573,0,New,Urban,Spring,2026-08-02T00:00:00.000Z,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2
AD_478798,Electronics,Tablet,Karnataka,Image,Carousel,1.97,0.0,2,236,0.00847457627118644,0.027966101694915254,3.94,0.0,0.0,0,0.0,0.0,42.847457627118644,236,0.0,0.09999999999999992,0.016117721830710197,"List(instagram, Unknown, google, facebook)","List(Unknown, Tablet, Mobile, Desktop)",0.004139926091193895,10036.52,2810.527602477398,"List(Afternoon, Evening_Prime, Late_Night, Morning)",5,11.572033898305085,2026-08-02,2026-08-02 10:11:18.245398,0.0,-1.0,0E-9,0,0.0,0.0033898305084745766,0.0016117721830710184,0,New,Urban,Winter,2026-08-02T00:00:00.000Z,2026-08-02T11:03:05.998933,20260802_110305,development,2026,8,2



Data types:
object            21
float64           19
int32              6
int64              4
datetime64[ns]     1
Name: count, dtype: int64

Basic statistics:


ad_video_length,total_clicks,total_impressions,ctr,avg_watch_duration,total_revenue,roas,total_conversions,conversion_rate,overall_conversion_rate,avg_user_age,unique_users,avg_watch_ratio,avg_ded_score,category_age_affinity,platform_avg_roas,platform_total_revenue,best_day,avg_hour,engagement_efficiency,profit_margin,high_performance,cost_efficiency_score,engagement_score,audience_alignment_score,ad_age_days,year,month,day
9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0,9999.0
11.996699669966997,3.456145614561456,212.52125212521253,0.016266933343716736,1.876384184050196,1.6738156491269673,0.22216152374312245,0.0959095909590959,0.023413061419362047,4.512356117336107E-4,41.531113476713806,212.06660666066605,0.06107657863456042,0.09999999999999992,0.01626396572777839,0.004139926091193895,2810.527602477399,3.8391839183918393,11.497291822447858,0.0012002902950602977,-0.7034310355128032,0.048804880488048805,2.7909888476359494E-4,0.03185366535366343,0.001626396572777838,0.0,2026.0,8.0,2.0
19.985894433429085,1.9951060309701543,14.657369855828751,0.009350958050129613,2.672473478232571,6.853577569788995,0.9273153249025617,0.3506092634231765,0.09275995417378445,0.0016523897427667283,0.9613911345399251,14.602191917590977,0.10835740847803582,1.9607007062439436E-17,0.0023044571533746895,0.0,9.09540184467517E-13,2.0685735600630903,0.4773653698894366,0.002505386811070089,0.9464287221960709,0.21547066471715806,0.0011532275391002603,0.05245406982951351,2.304457153374688E-4,0.0,0.0,0.0,0.0
0.0,0.0,153.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.87980769230769,153.0,0.0,0.09999999999999987,0.009607903856231576,0.004139926091193895,2810.527602477398,1.0,9.560975609756097,0.0,-1.0,0.0,0.0,0.0,9.60790385623157E-4,0.0,2026.0,8.0,2.0
0.0,2.0,202.0,0.009389671361502348,0.026501677640707384,0.0,0.0,0.0,0.0,0.0,40.87926869898384,202.0,0.0,0.09999999999999991,0.01532488428989158,0.004139926091193895,2810.527602477398,2.0,11.175902661455424,0.0,-1.0,0.0,0.0,0.004020100502512563,0.001532488428989157,0.0,2026.0,8.0,2.0
0.0,3.0,213.0,0.01485148514851485,0.054245283018867926,0.0,0.0,0.0,0.0,0.0,41.53909465020576,212.0,0.0,0.09999999999999992,0.01689434154234005,0.004139926091193895,2810.527602477398,3.0,11.50253807106599,0.0,-1.0,0.0,0.0,0.007766990291262136,0.0016894341542340041,0.0,2026.0,8.0,2.0
15.0,5.0,222.0,0.022222222222222223,5.572193585337915,0.0,0.0,0.0,0.0,0.0,42.177570093457945,222.0,0.1004585326953748,0.09999999999999994,0.017972565534661908,0.004139926091193895,2810.527602477398,6.0,11.815384615384616,0.0016379432481487917,-1.0,0.0,0.0,0.03992448471926083,0.0017972565534661892,0.0,2026.0,8.0,2.0
60.0,14.0,268.0,0.07407407407407407,7.933802816901407,85.60091743346959,14.608566511036615,4.0,1.0,0.02,45.34146341463415,268.0,0.4231391585760518,0.09999999999999999,0.019878129960146013,0.004139926091193895,2810.527602477398,7.0,13.105504587155963,0.023200126023944553,13.608566511036614,1.0,0.031746031746031744,0.4190678733031674,0.0019878129960146005,0.0,2026.0,8.0,2.0



TARGET DISTRIBUTIONS

CTR Distribution:
  Count: 9,999
  Mean: 0.0163
  Min: 0.0000
  Max: 0.0741
  Std: 0.0094

ROAS Distribution:
  Count: 9,999
  Mean: 0.2222
  Min: 0.0000
  Max: 14.6086
  Std: 0.9273

CONVERSION_RATE Distribution:
  Count: 9,999
  Mean: 0.0234
  Min: 0.0000
  Max: 1.0000
  Std: 0.0928

HIGH_PERFORMANCE Distribution:
  Count: 9,999
  Mean: 0.0488
  Min: 0.0000
  Max: 1.0000
  Std: 0.2155

High Performance Class Balance:
  0 (Not Profitable): 9,511 (95.1%)
  1 (Profitable):     488 (4.9%)

--------------------------------------------------
CLASS IMBALANCE RECOMMENDATIONS
--------------------------------------------------
  Imbalance Ratio: 19.5:1
  Recommended approaches for classification models:
    1. Use class_weight='balanced' in RandomForest/GradientBoosting
    2. Use scale_pos_weight in XGBoost
    3. Use SMOTE for oversampling (with caution on 1,000 rows)
    4. Use StratifiedKFold for cross-validation

TRAIN/TEST SPLIT
Class distribution in full dataset:


ctr,roas,conversion_rate,high_performance,total_impressions,total_clicks,avg_watch_ratio,avg_ded_score
1.0,0.07637096820480847,0.07855411714951856,0.07729526711373635,-0.00555666926973123,0.9905856986968293,0.19846836473474513,0.0020813611498070503
0.07637096820480847,1.0,0.9481990389927207,0.8599211993563651,0.004940763437804203,0.07570513910373251,0.43684952059875637,-0.008549423421745339
0.07855411714951856,0.9481990389927207,1.0,0.8246159848474213,0.0014195637323783103,0.07765104930491938,0.4591987074216792,-0.007811134460857345
0.07729526711373635,0.8599211993563651,0.8246159848474213,1.0,0.0019974937912058677,0.07592201146294424,0.4261493591000768,-0.007318791395828931
-0.00555666926973123,0.004940763437804203,0.0014195637323783103,0.0019974937912058677,1.0,0.11289821283922874,0.007619663680260093,-0.466791779915074
0.9905856986968293,0.07570513910373251,0.07765104930491938,0.07592201146294424,0.11289821283922874,1.0,0.1975656083937505,-0.05451973131748005
0.19846836473474513,0.43684952059875637,0.4591987074216792,0.4261493591000768,0.007619663680260093,0.1975656083937505,1.0,-0.00919687214042345
0.0020813611498070503,-0.008549423421745339,-0.007811134460857345,-0.007318791395828931,-0.466791779915074,-0.05451973131748005,-0.00919687214042345,1.0



Top correlations with ctr:
  total_clicks: 0.991
  avg_watch_ratio: 0.198
  conversion_rate: 0.079
  high_performance: 0.077
  roas: 0.076
  avg_ded_score: 0.002
  total_impressions: -0.006

Top correlations with roas:
  conversion_rate: 0.948
  high_performance: 0.860
  avg_watch_ratio: 0.437
  ctr: 0.076
  total_clicks: 0.076
  total_impressions: 0.005
  avg_ded_score: -0.009

Top correlations with conversion_rate:
  roas: 0.948
  high_performance: 0.825
  avg_watch_ratio: 0.459
  ctr: 0.079
  total_clicks: 0.078
  total_impressions: 0.001
  avg_ded_score: -0.008

CATEGORICAL FEATURE ANALYSIS

AD_CATEGORY Distribution:
ad_category
Electronics    1518
Fashion        1323
Health         1319
Food           1301
Gaming         1274
Travel         1264

AD_DEVICE Distribution:
ad_device
All-Devices    2385
Mobile         2103
Desktop        1757
Tablet         1754

AD_TYPE Distribution:
ad_type
Image      2660
Video      2658
Text       1363
Unknown    1318

AD_LOCATION Distribution:
a

In [0]:
# ============================================================
# VERIFY NEW SPLIT FILES
# ============================================================

import pandas as pd

volume_path = "/Volumes/adtech_catalog/bronze/landing_zone/"

train_df = pd.read_csv(volume_path + "train_split.csv")
test_df = pd.read_csv(volume_path + "test_split.csv")

print("="*70)
print("VERIFYING SPLIT FILES")
print("="*70)

print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")

train_hp = train_df['high_performance'].value_counts()
test_hp = test_df['high_performance'].value_counts()

print(f"\nTraining class distribution:")
print(f"  Class 0: {train_hp.get(0, 0)}")
print(f"  Class 1: {train_hp.get(1, 0)}")

print(f"\nTest class distribution:")
print(f"  Class 0: {test_hp.get(0, 0)}")
print(f"  Class 1: {test_hp.get(1, 0)}")

if train_hp.get(1, 0) > 0:
    print("\n  BOTH CLASSES PRESENT IN TRAINING SET ")
else:
    print("\n  ERROR: Training set has ONLY class 0 ")

VERIFYING SPLIT FILES
Training rows: 7,999
Test rows: 2,000

Training class distribution:
  Class 0: 7609
  Class 1: 390

Test class distribution:
  Class 0: 1902
  Class 1: 98

  BOTH CLASSES PRESENT IN TRAINING SET 
